# Notebook 00 - Welcome and Quick Result

Start here for the shortest complete Tuba workflow:

1. Build a small piping model.
2. Run or import Code_Aster result artifacts.
3. Open an interactive deformed-stress visualization in the notebook.

The result cells do not synthesize stress, displacement, or reaction values. If the expected Code_Aster tables are missing, the notebook runs the configured Code_Aster runtime. If that runtime is unavailable, it stops before displaying solver results.


## Notebook Setup


In [ ]:
import sys
from pathlib import Path

# Ensure the repo root is on sys.path so `import tuba` works
# regardless of where the kernel was started.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pyvista as pv

from tuba import Model
from tuba.analysis.code_aster_notebook import configure_code_aster_notebook_runtime, load_or_run_code_aster_results
from tuba.plotting import plots
from tuba.plotting.notebook import configure_notebook_backend

# Defaults to zoomable embedded HTML locally; set TUBA_NOTEBOOK_BACKEND=client or static to override.
JUPYTER_BACKEND = configure_notebook_backend()

print(f"Tuba v4 loaded from: {REPO_ROOT}")
print(f"PyVista version:     {pv.__version__}")
print(f"Notebook backend:    {JUPYTER_BACKEND}")


## Build the Demo Model

This is the complete model needed for the first solve: material, section, geometry, support, and operating load case.

Course vocabulary used throughout the notebooks:

- Coordinates are plain `[x, y, z]` values in meters. You do not need `np.array(...)` for public model-authoring examples.
- A node id such as `N0` points to one coordinate in `model.nodes`.
- An element connects two node ids. `n1` is the start endpoint, `n2` is the end endpoint, and the element vector is `model.nodes[n2].coords - model.nodes[n1].coords`.
- `PipingBuilder` creates most pipe nodes and elements for you. Direct `add_node` / `add_element` calls are only needed when a notebook builds frame, beam, or imported-component structures explicitly.

In [ ]:
model = Model("StressDemo", standard="ASME_B31.3")

model.add_material(
    "P265GH",
    E=2.0e11,
    nu=0.3,
    rho=7850.0,
    alpha=1.2e-5,
    allowable_stress={20.0: 137e6, 200.0: 120e6},
)
model.add_pipe_section("4inch_sch40", OD=0.1143, WT=0.00602, corrosion_allowance=0.001)

with model.pipe(section="4inch_sch40", material="P265GH") as b:
    b.start([0, 0, 0], support="anchor")
    b.run(5.0)
    b.bend(radius=0.2, angle=90.0, plane="XY")
    b.run(3.0)
    b.end(support="anchor")

model.add_support("N2", type="guide")
model.define_load_case("Operating", gravity=True, pressure=1.5e6, temperature=200.0)

print(f"Project:   {model.project_name}")
print(f"Nodes:     {len(model.nodes)}")
print(f"Elements:  {len(model.elements)}")
print(f"Supports:  {len(model.supports)}")


## Run Code_Aster and Show the First Result

Run this cell to get to the interactive engineering view. It reuses existing Code_Aster result tables when they are present; otherwise it executes Code_Aster through the configured runtime and then imports the generated artifacts.


In [ ]:
CODE_ASTER_RUNTIME = configure_code_aster_notebook_runtime()
# VS Code/Jupyter review defaults to committed real Code_Aster artifacts; set True only after the runtime doctor passes.
RUN_CODE_ASTER = False
CODE_ASTER_WORK_DIR = REPO_ROOT / "notebooks" / "code_aster_results" / "stress_analysis_operating"

code_aster_run = load_or_run_code_aster_results(
    model,
    "Operating",
    CODE_ASTER_WORK_DIR,
    run_solver=RUN_CODE_ASTER,
    exec_method=CODE_ASTER_RUNTIME.exec_method,
    wsl_distro=CODE_ASTER_RUNTIME.wsl_distro,
    docker_image=CODE_ASTER_RUNTIME.docker_image,
)
results = code_aster_run.results
code_aster_artifact = code_aster_run.artifact

source = "ran Code_Aster" if code_aster_run.ran_solver else "loaded existing Code_Aster tables"
print(f"Result source: {source}")
print(f"Work directory: {CODE_ASTER_WORK_DIR.resolve()}")
print(f"Node results: {len(results.node_results)}")
print(f"Element results: {len(results.element_results)}")

plots.plot_deformed_stress(results, deform_scale=100.0, model=model, jupyter_backend=JUPYTER_BACKEND)


## Optional: Installation

Tuba v4 requires Python 3.10+.

Core install from the repository root:

```bash
pip install -e .
```

Notebook visualization dependencies:

```bash
pip install -e ".[course]"
```

pip installs Tuba, not Code_Aster. The solver is a separate Linux runtime. On Windows the tested path is WSL2 Ubuntu; see `docs/code_aster_installation.md`.

Before setting `RUN_CODE_ASTER = True`, verify the runtime from the repository root:

```powershell
python -m tuba.solver.code_aster_doctor --check
```


## Optional: Architecture Overview

At the centre of Tuba v4 sits the **`TubaModel`** (aliased as `Model`).  
It is the *single source of truth* for every piping system — geometry, properties, loads, and boundary conditions are all stored in one object.

```
┌─────────────────────────────────────────────┐
│                  TubaModel                  │
├─────────────────────────────────────────────┤
│  Materials      dict[str, Material]         │
│  Sections       dict[str, PipeSection|...]  │
│  Nodes          dict[str, Node]             │
│  Elements       list[Element]               │
│  Supports       dict[str, Support]          │
│  Load Cases     dict[str, LoadCase]         │
│  Tees           dict[str, TeeDefinition]    │
│  Obstacles      list[Obstacle]              │
└─────────────────────────────────────────────┘
```

The entire model can be round-tripped through JSON:

| Method | Description |
|---|---|
| `model.to_dict()` | Convert to a plain Python dictionary |
| `model.to_json(path)` | Write JSON file to disk |
| `Model.from_dict(d)` | Reconstruct from dictionary |
| `Model.from_json(path)` | Load from JSON file |

This makes every Tuba model fully **serialisable, diffable, and AI-parseable**.

## Optional: Inspect the Model JSON

The canonical model representation is JSON. Use this after the first result view when you want to inspect exactly what was solved.


In [ ]:
import json

model_dict = model.to_dict()
print(json.dumps(model_dict, indent=2)[:2500])
print("... truncated ...")


## Optional: Geometry-Only Preview

This preview is useful for checking raw geometry. It is not a solver result view; the stress/displacement result view above comes from Code_Aster artifacts.


In [ ]:
from tuba.plotting.pipeline import build_3d_mesh_from_model

# True section geometry straight from the model — the OD/wall come from the
# pipe section, so this preview can never disagree with what gets solved.
tube_mesh = build_3d_mesh_from_model(model)

# --- Render ---
plotter = pv.Plotter()
plotter.set_background("#1a1a2e")
plotter.add_mesh(tube_mesh, color="#5c6b73", smooth_shading=True)
plotter.add_axes()
plotter.camera.azimuth = 30
plotter.camera.elevation = 20
plotter.show(jupyter_backend=JUPYTER_BACKEND)

## Course Map

You have now built a Tuba model, loaded or ran Code_Aster, and opened the first interactive result view.

| Notebook | Role |
|---|---|
| **00** | Fast complete workflow: model -> Code_Aster -> interactive result |
| **01** | Geometry authoring with the `PipingBuilder` DSL |
| **02** | Supports, boundary conditions, and load cases |
| **03** | Code_Aster-backed stress analysis and ASME B31.3 compliance |
| **04** | Result visualization and export formats |
| **05** | Deterministic autorouting and Code_Aster study handoff |
| **06** | Coupled pipe racks and Code_Aster-backed design evaluation |
| **07** | JSON and IFC/BIM data exchange with solver properties |
| **08** | Expansion-aware hot-line autorouting and envelope review |
| **09** | Imported CAD component placement and pipe coupling |
| **10** | Interactive Code_Aster artifact post-processing |

Supplemental notebooks: `autorouting_quick_iteration.ipynb`, `visualize_elements_and_supports.ipynb`, and `advanced_piping_design_and_bim.ipynb`.